

---Summary: In this lesson, we'll use a pre-trained Multi-task Cascaded Convolutional Network (MTCNN) for face detection. We'll load the model from facenet_pytorch library and use it on images extracted from Mary Kom YouTube interview video. We'll demonstrate a variety of capabilities that a pre-trained MTCNN model comes with.

Objectives:

Initialize a pre-trained MTCNN model from facenet_pytorch
Detect faces in an image using MTCNN model
Display the resulting bounding boxes of faces detected by the model
Crop out detected faces for further analysis
Determine facial landmarks such as eyes, nose, and mouth using the MTCNN model
Select a subset of images for face recognition tasks in the next lesson
New Terms:

Face detection
MTCNN
Facial landmarks



In [ ]:
import shutil
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import PIL
import torch
import torchvision
from facenet_pytorch import MTCNN
from PIL import Image
from torchvision.utils import make_grid

In [2]:
pip install facenet_pytorch

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 50.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 132.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 142.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 755.5/755.5 MB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 410.6/410.6 MB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.1/14.1 MB 93.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 70.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 823.6/823.6 kB 48.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 731.7/731.7 MB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.6/121.6 MB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 MB 15.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 124

In [2]:
import shutil
import sys
from pathlib import Path
import matplotlib.pyplot as plt
import PIL
import torch
import torchvision
from facenet_pytorch import MTCNN
from PIL import Image
from torchvision.utils import make_grid

In [4]:
if torch.cuda.is_available():
  device="cuda"
elif torch.backend.mps.is_available():
  device="mps"
else:
   device="cpu"

In [5]:
MTCNN?

Initializing a MTCNN model
We'll perform face detection using a MTCNN network from facenet_pytorch library. This model is able to simultaneously propose bounding boxes of faces, determine detection probabilities, and detect facial landmarks like eyes, nose and mouth.

Let's start by initializing the model. Here are a couple of arguments we get to set:

device: The device on which to run the model.
keep_all: A boolean determining if all detected faces are returned or not.

min_face_size: Minimum face size (in pixels) to search for in the image.

post_process: A boolean determining if we want image standardization of detected faces. This is advised before proceeding with face recognition models, but if we want face images that are returned to us to look normal to the human eye, we can set post_process=False.

**Task 4.3.1: Initialize a MTCNN model. Make sure to use a GPU, keep all detected faces and set minimum face size to search for to be 60.**

In [6]:
mtcnn = MTCNN(device=device, keep_all=True, min_face_size=60, post_process=False)

print(mtcnn)

MTCNN(
  (pnet): PNet(
    (conv1): Conv2d(3, 10, kernel_size=(3, 3), stride=(1, 1))
    (prelu1): PReLU(num_parameters=10)
    (pool1): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=True)
    (conv2): Conv2d(10, 16, kernel_size=(3, 3), stride=(1, 1))
    (prelu2): PReLU(num_parameters=16)
    (conv3): Conv2d(16, 32, kernel_size=(3, 3), stride=(1, 1))
    (prelu3): PReLU(num_parameters=32)
    (conv4_1): Conv2d(32, 2, kernel_size=(1, 1), stride=(1, 1))
    (softmax4_1): Softmax(dim=1)
    (conv4_2): Conv2d(32, 4, kernel_size=(1, 1), stride=(1, 1))
  )
  (rnet): RNet(
    (conv1): Conv2d(3, 28, kernel_size=(3, 3), stride=(1, 1))
    (prelu1): PReLU(num_parameters=28)
    (pool1): MaxPool2d(kernel_size=3, stride=2, padding=0, dilation=1, ceil_mode=True)
    (conv2): Conv2d(28, 48, kernel_size=(3, 3), stride=(1, 1))
    (prelu2): PReLU(num_parameters=48)
    (pool2): MaxPool2d(kernel_size=3, stride=2, padding=0, dilation=1, ceil_mode=True)
    (conv3): Conv2d(48, 64,


**Task 4.3.2: Create a variable for the current working directory using pathlib syntax.**

In [7]:
curr_work_dir = Path.cwd()

print(curr_work_dir)

/content



**Task 4.3.3: Create an absolute path for the extracted_frames directory using the pathlib syntax**

In [8]:
extracted_frames_dir = curr_work_dir /"project4" / "data" / " extracted frames"

print(extracted_frames_dir)

/content/project4/data/ extracted frames



**Task 4.3.4: Create a file path to the sample image we'll be working with. The image is in the extracted_frames directory.**

In [1]:
sample_image_filename = "frame_320.jpg"
sample_image_path = extracted_frames_dir / sample_image_filename

sample_image = Image.open(sample_image_path)
sample_image

NameError: name 'extracted_frames_dir' is not defined


Bounding Boxes of Detected Faces
If we want to detect faces and obtain their bounding boxes, we need to use the detect method on the MTCNN model and pass in the sample image. This returns both the bounding boxes of detected faces as well as the predicted probability that the object in a given bounding box is indeed a face.

**Task 4.3.5: Use the detect method on the MTCNN model we initialized in one of the previous tasks. Make sure to pass in the sample_image.**

In [ ]:
boxes, probs= mtcnn.detect(sample_image)
print("boxes type:", type(boxes))
print("probs type:", type(probs))

In [ ]:
print(boxes)
print(boxes.shape)


**Task 4.3.6: Using boxes, compute how many faces were detected in the sample image.**

In [ ]:
number_of_detected_faces = len(boxes)

print(number_of_detected_faces)

In [ ]:
sample_image


**Task 4.3.7: Using probs, determine for how many of the faces detected did the model predict with at least 99% probability that it's a face.**

In [ ]:
num_faces = len(probs[probs>0.99])

print(num_faces)

Looks like the model is very certain that all of the detected faces are indeed faces!

Now let's plot the bounding boxes together with the sample image.

**Task 4.3.8: Fill in the missing code below to iterate over all of the bounding boxes and plot them on top of the sample image**

In [ ]:
fig, ax = plt.subplots()
ax.imshow(sample_image)

for box in  boxes:
    rect = plt.Rectangle(
        (box[0], box[1]), box[2] - box[0], box[3] - box[1], fill=False, color="blue"
    )
    ax.add_patch(rect)
plt.axis("off");


**Extracting Facial Landmarks**
MTCNN not only detects faces but can also mark facial landmarks such as eyes, nose, and mouth in each detected face.

The way to obtain the facial landmarks together with bounding boxes and probabilities is to again use the detect method on the MTCNN model. But this time together with the sample image, we need to pass in landmarks=True.

**Task 4.3.9: Use the detect method on the MTCNN model such that we'll get bounding boxes, probabilities and facial landmarks returned.**

In [ ]:
boxes, probs, landmarks = mtcnn.detect(sample_image, landmarks=True)

print("boxes type:", type(boxes))
print("probs type:", type(probs))
print("landmarks type:", type(landmarks))

The facial landmarks detected by the model on each face are:

left eye,
right eye,
nose,
left mouth corner,
right mouth corner.
Let's make sure that the shape of the landmarks array matches what we'd expect given that six faces were detected.

**Task 4.3.10: Print the shape of the landmarks array returned by the model.**

In [ ]:
print(landmarks.shape)

**Task 4.3.11: Fill in the missing code to plot the bounding boxes as well as the facial landmarks on top of the sample image. We recommend using zip on boxes and landmarks in the for loop that you need to fill in.**

In [ ]:
fig, ax = plt.subplots()
ax.imshow(sample_image)

for box, landmark in  zip(boxes, landmarks):
    rect = plt.Rectangle(
        (box[0], box[1]), box[2] - box[0], box[3] - box[1], fill=False, color="blue"
    )
    ax.add_patch(rect)
    for point in landmark:
        ax.plot(point[0], point[1], marker="o", color="red")
plt.axis("off");

Cropping out Detected Faces
If we wanted to proceed with further face analysis like for example perform face recognition, it's a good idea to crop out the detected faces. That way further analysis can focus only on the relevant parts of the image.

So let's learn how we can crop out the detected faces!

In order to get the PyTorch tensors of the detected faces instead of the bounding boxes, we need to call the MTCNN object directly and just pass in the image we're working with.

**Task 4.3.12: Use the MTCNN model that we initialized in the first task and pass it the sample_image.**

In [ ]:
faces = mtcnn(sample_image)

print(faces.shape)

Looks like this returned three small images, each with 3 color channels and 160 width and 160 height. Let's plot these 3 images!

**Task 4.3.13: Create a grid of these three images by using make_grid from torchvision.utils and passing in faces. Use nrow=3 so we'll have all 3 images in one row.**

In [ ]:
Grid = make_grid(faces, nrow=3)

print(Grid.shape)

In [ ]:
plt.imshow(Grid.permute(1, 2, 0).int())
plt.axis("off");

vPrepare a Subset of Images for Next Steps
To conclude this lesson, we'll prepare a subset of images for the next lesson where we'll work on face embeddings or faceprints. These are numerical representations of a face that are needed for tasks like face recognition or verification.

Let's create a directory of selected images that we'll work with in the next lesson.
**bold text**
**Task 4.3.14: Make a directory into which we'll put the selected images. Make sure you do it such that no error is raised even if the directory already exists. **

In [ ]:
images_dir = curr_work_dir / "project4" / "data" / "images"
images_dir.mkdir(exist_ok=True)

**Task 4.3.15: Make a subdirectory in the images directory and call it mary_kom. Again make sure you do it such that no error is raised even if the directory already exists.**

In [ ]:
mary_kom_dir = images_dir / "mary_kom"

# Now Create `mary_kom` directory
mary_kom_dir.mkdir(exist_ok=True)

In [ ]:
mary_kom_imgs = [
    "frame_80.jpg",
    "frame_115.jpg",
    "frame_120.jpg",
    "frame_125.jpg",
    "frame_135.jpg",
]


**Task 4.3.16: Iterate over mary_kom_imgs list of image filenames and create a list of absolute paths to each image using pathlib syntax. Remember that the images are in the extracted_frames directory**

In [ ]:
mary_kom_img_paths = [extracted_frames_dir / i for i in mary_kom_imgs]

print("Number of images we'll use:", len(mary_kom_img_paths))

In [ ]:
fig, axs = plt.subplots(1, 5, figsize=(10, 8))

for i, ax in enumerate(axs):
    ax.imshow(Image.open(mary_kom_img_paths[i]))
    ax.axis("off")


**Task 4.3.17: Iterate over mary_kom_img_paths in order to copy these selected images into mary_kom directory.**

In [ ]:
for image_path in  mary_kom_img_paths:
    shutil.copy(image_path, mary_kom_dir)

In [ ]:
print("Number of files in mary_kom directory:", len(list(mary_kom_dir.iterdir())))

We'll also get some images of the interviewer, so we'll have more than one face we can potentially identify. We'll call that directory ranveer, since that's the interviewer's first name.

**Task 4.3.18: Make a subdirectory in the images directory and call it ranveer. Again make sure you do it such that no error is raised even if the directory already exists.**

In [ ]:
ranveer_dir = images_dir / " ranveer"

# Now Create `ranveer` directory
ranveer_dir.mkdir(exist_ok=True)

In [ ]:
ranveer_imgs = [
    "frame_10.jpg",
    "frame_40.jpg",
    "frame_270.jpg",
    "frame_365.jpg",
    "frame_425.jpg",
]


**Task 4.3.19: Iterate over ranveer_imgs list of image filenames and create a list of absolute paths to each image using pathlib syntax. Remember that the images are in the extracted_frames directory.**

In [ ]:
ranveer_img_paths = [ extracted_frames_dir / i for i in ranveer_imgs]

print("Number of images we'll use:", len(ranveer_img_paths))

In [ ]:
fig, axs = plt.subplots(1, 5, figsize=(10, 8))

for i, ax in enumerate(axs):
    ax.imshow(Image.open(ranveer_img_paths[i]))
    ax.axis("off")

**Task 4.3.20: Iterate over ranveer_img_paths in order to copy these selected images into ranveer directory.**

In [ ]:
for image_path in ranveer_img_paths:
    shutil.copy(image_path, ranveer_dir)

print("Number of files in ranveer directory:", len(list(ranveer_dir.iterdir())))